# GwenLand glcuda Wave 118 - in-process T4 stability gate

One model/context, ABBA production prefill, two reversed invocations.


In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import shutil
import statistics
import subprocess
import traceback
import urllib.request
import zipfile

BUILD = "wave118-in-process-stability-v1"
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "5de5be39c0190b9367da18f0f2001e7f40208c92"
SOURCE_REV = "d9d540764054a233aca9b4b74af5ea22b144f54e"
PATCH_SHA256 = "0412028f99fdfe9dc86d31e71e0df68ccbe6ad7571174cd5656d24ebd66e4fd2"
PATCH_GZIP_B64 = """H4sIAJD8pWoC/7Va63LbOLL+76dAtFVeakzRulA3epwZJ/GksjuOs7azs1WOi4ZIUMKaF4UgbWsdV52HOE94nuR0A7yAkmwnU7X+YUkg0Gj09esGfR4EpNOZ84zQ/Xno5T7dZ/c0WoZM7N/RW9brTVweu8s08ZgQrsjojIc8W1mpILMfXbETszsS8JCRKPEZ6XW7I9ve4bHP7kn3O/8sq9+d9P2JPfK69nA2GQ4HI+oFIzqawc7jYX80GAw9n43tnU6nQ/Z9drsf52G4s7e39yc4/vVX0umaXbLXM3v9Hvn11529/f1X5A9YRmAd4XGnWEfg08+9jCcxqUiQBU1jeGjJZWrtaazOH5okzeOYpSZ5+/ndEaznEU1XxEvijN1nJqGxT2gYJh6VRGmYsTSmGSPZgilSKcsoj5kvp/osYGnK/E7KBPdzGpIlzRbCIm+TKIL1Gn9BSOeC0JQRkS+XIWe+ojdbIW3iwa4sJTMWJDClPB8cKs0Oigm5APrijmfegnBB2D1Q8cCKFixlcNidvVwwAsIGAo7D4jlw6WYp5ZnjPLwPj+WAST7EwPKHeJlnjwfVEtCPnIRf3iZxwOcmUb/UsnoqcuY47+WnenaAW4MARUY+nZ2efLpwP3/8cOGQXZGl5JC0ThgVeYoSBKZ9BgKNeMxFxj0iViJjkVRjtMzgiCkLwG5WFjmGw4GYySK5I1lyw0DlNEURhaD+jM1BVBHNUn5PojzMOEpCaQy4BLGBCYCGIhYl6cokFywWSQo6AS0pFQf8Hp6HNI9BlnOWRCxLYdfWQXmSf3w+Ors4vjh3gCD/D4NzDLvVwz+Ozk4+fzp3Px2fufBVm1NNOf7Xp+O3F8fv3EIkF6d/P/6oUevbtpRbEMN5QBcG9wWI7DIf9K/apPNaUxN52Nkj8Lc5gn9SOK5cDf+sLHFvmWe0zXpGRO9dCAKunAnTetozkP6SpTQD/Tika3X1R8nSvXGIvT62xInToTaasiWjmbtkMbjLCjaw9C0sq2bcccBhKCjMaKsJjzt7j4UYwC9dmkaGeqDMFySiW2FBVRNVMeKBTrkPbuqQWZKExWiSUi+EIZgII1KqZ0zA7j8HI9skb5L7n/0VBg4f3CVNk9RxjvHj9etSwIoLS7DMnTEwFYgVN670eTcIYrd0eqPav/3LgVoZsowkoKnDkgZHIRiVrtvVTB7gREu5QKEk8urwCQsi377J6ZXaLQrRE6yfGW1cdakOfaWbCESsPI0JnM2A4AJu88qoH+JfSy0iERfw1Fs4EKaiw4dH0mAKB8pvzi+PGH+YlzH/8PLh8aplNknCqSqhkAfSqn60CKwMhRwsQymMrS3fkMiW55UI1p/Jw2iDbc0YIXgkRrt9UJkffpzeGGpDBokydCPRrs0yAg4N3XbAt77HdNAAIjAAmkLUPyxmxreOgwNG2xI3fGn02pq5yPwEU3GCxnEMecnQj5DcuElqtCAbzMG4n0um5OT03fHv5JJ+m121GqaZsluWCgxEUuFMvDJw22IztCkfkkqAZz2H4Gi0Zi0ptJpCEbIP9ZhvqUhgTBrHAiHMlznM1F3ZcSCPLVxP5hpDTzy65QrGfEdxYPf1mPacNxYLAgpWVq55LDkCTsAGOBzyF20kTKjvSvkbu/KjISwwMeAe58Gu8NT1FnDIXSUB3Y/RHUMWK098wn9/yC/LvAjoBIw6I37Kg8wh4IiwwcPjhtOV+6+Nb2flOx2kqUmv0GSd/h0HkotRLvA25Ou9KF/lsK4Kl4pAHSt3G8Gyng5Tf6oXageowoI2FvBUbHUihWdKknMWYzIEZBAnKtS1GscQizzzk7vYqF0BFKcFOgAsl9LuTJKleTMK40wXZwCcXgMQ+jRpGEUq3AWbMwlKwKw3MQtuK860QNZQV4w2hQFFZ6n8L4euNiOCj/O1KdWHPE9DCxAhYDKYfRlMHmoiVYwvmHjU3Sm+LeH12nKIMlp2oK1qGcruaw5QWC5HCZb4bF3CxjIRHGlrEmvjGsmvtG6XAwSFIMfiPJLqBn9dU4AKcGU2ADZ/XCMlGViljvlD6XCNEJQpcRbG6/FBxojLIgN0hCyvwOYevrRqGX9pOV9aD49fWuaXViFCGHp4hJ+lqMrfwKo+u5F/qzWVVHDEsab6YLYsRkc4WninXC+XP24ELBm0Kla3PCw43vKk0vPmIzjHtn2eCI5KvuWpnltIqCCAHMlPWEUDWCb7zy/cACIy1m712ieUXCsXXMTnXtaRQfE7dFwpTFmFAHSeukrBhR63KFcxXFDbR4JNlW1VVemJW4ZAUn19+RYNaFPt54f1DN4IxAWCg6SFqM3f0l0Rqbd/A3U8ONo+pJ9GH2XzWdEcmfXtYDqbUm/UGw2m3Zk/m3ThRzAY9Gfj3mDGJv7UmwRDyxpNJvY08PzeZEw9exIMhn3me96IjYcwYg+mwy6ze11aNl+wR/IMb82uyZbn2B8ZmSOyNzLH2B0h2EZIYo8RUDJGwTua+mRJhSAGhEQczJjI2tYO2cH6WIHRIOCO47m3CfeL4l4Oi1UM9T3NkojD58OR/PIGiypyiiGUx3OITgWhstnwPpToF8eRuX5/CJzt9fuKQbLMZ0A8zQHB/F0e5ZxhDSs1PWdR5Ma9kYuuxGTxoUo4+XR/f7/s+/Qc1W0hjELNHidxJ+BQcJLffvtIqu4LxnfZLUEgC/X9iqV/FTUpGqaM+qtOkGOSolnGYpmHzk7O9/4xkSKzyOkS3Yzk8CjUGzhH+2+snQ6S2oY5tbpz22NNjusnGx5hN2e9rYRV8oHqmYBGAdfLY6XJHVHWQGbUu9Fo3S24agvdATgFIsguoXOgKSBnAhoAKC0sNR+oCFdS9UvGpdqmA7PXA71NbbM/fUlxgQvygyMira83NlYfM6hH1Dxg7BGMQdb1TCThLdtWNhfJHZThfp248zBv1u9QLvE0iSNQkstiOgtrdveaRUAC4THlPiwH5YGufsZZr8uiH3+UGV7fjOzubqFh5fFdSpcID7cwUFSGhENU3ZQJKuJvHy6kqlg0Y74PCv108S+p1EIQBOHOCg6Xwf9lAhYLJgelZOSOh3uyzViRymTfqoM+htVhDhAVFIxgGshmSUKM6/e/Yx/T/XjqnpwcHfauSbLMBJbHJppNTQqtoTor+HBlGsgq7t0l/tKm5P3xyUm7MBPUPigQ9zNkh5DsvoUPvRiuJIDtQlUB61jqnIWB40j0L8s9ZQOlpCVRk3xMYtZew6/I8u+wTLJXGLzIAbgRpEMoiWmKvlAdqZPE4YqUlCUURO+qqUEAscg17nWNQpytMtaBWR38InfRPF3TO2go5N7KUoT+cuknnrHgoNe4fVWk7VpKWw9ZS0OToZbqNv3iaXOWefBF8TfwZ2E3h3L3qhCDIQPsEhFrY7ZAsCr1gkVYYoFlRPTfSWqS5hgHXN9eWxtFVCFdIPIaqIxNMmyjj9W9j1sKghFGq2G1rbbFhQsBXQFxjESDycSckD27O8aABCPb3U2aTAGfdxrQC/WszTlQcbvkdFPk5LCesS1MPHmEd8e/QQ0HOcg9Oz7/8O7z0e/qOAJbEDrWe2rfl+PjNp7WQOWfZG+NyjOWqHfT1tSuJZNGn2uNk6OLC2Di9I/zpnzWLch266x8iAP15Gf1IMmDMdkvkU/Z/Ku+h7Q3uzdFwGLbPQVYnjY3HbKY24ax17x1dgVwzKapbUr9RbXocEK1XzbntNfY0NS0zuBXOsaWFscua+NJUx8KJPTGPRtl1RsPRmZv8rS0JMBZMAitqYbWioJJQTNBIroqUB3M42kTzFHfl4BOS4glstsO3Rppa4s1Fznc2BWQlzaggWo4hoG1uVLGTaPEv45zxkJ6L8HARtY6V/dzJWiFDJXdMQb5HXD1ApIK/w/4SZW2CPYiZEUlrIpGTexTnZPktVJa/IgRQ8grQyEhokUuQDLVBWlxAUqgQhKE1uQEMA+5QN2A7stG3H5x8ykRirw7k5AToQEgZZIE5VVkZx4mMxrWxPQkGeWZPMTLafLFqxSpG5M08F5b98AnVNT0GUtkAJoMOXc9lhek63qm1mcRMBC6VjZXSbY/BvDhs84cAmIB32QZUVzg+im/xWqDJJ6XL2nsrcjXHJCepbD1eDDGkq0/HnfN3rAE14iPkyVzM2RJYPOquCI0yQLk7/o8qgaClH11Z1QALAgGfWm+xj+Z9zP8eG2Sf0K+A43KSk+U8pJVXQ4FoOP8hMVZoR2cU6gFGChvEMq77M3o70JELW+bXTAUF0QK2nbn2Cttds/ADVmavTJePZPWGu1Fif70RFlSeJGA+v/n1xcMyMsCHGlvJfPqO/nQ7hx+mE6jP7uFH1lx6PaIJQWP6BzKglwgrvBCfLGA3VMvC5U5avWFqINzTaZgR5D/+5//hUgiL4HU/UIVspCMtczu1esjGCBkbFDwUas8AfRi7x4rTFlZkPefPltPNmJCPttowKixovEyHQdsOpvQYBxM7cFgxHp922azfnc6nHX9yXBsj/oT5g0ti3apH4wmw649HHj2tD8ddz02GfTHlPo9OuzbfsCCGR082Xgp9t1ouBTj6LjDHrot/O8N0GmlR8mWGvjUw7n6BkpTX9QNFvZHtMp57XKrkttv8h2Es4/v5TWXLFlSpqoQDtFAbQMCPSjLlkOS8Yh1cDbztWSnbsmKUiEf2WWlgJu8eaFAwhz1V1G9x9JMwPguy7a0VPADuRxs+7ZoTTxRP5V3VypFAeP4oga++7LW3qmzlsxC1+gG10QkkACw4EsjgR10IMfuuZC3AMBDmZxqaqUxqowE1thMXw3WAEJyGXdfSFvPXzY2SzSVO5ATNY5SZGlH9lllwsa2DcgL+1BYxAMkyOcLcnndvByFFH19pTLHdIoG2Os20JY+eyM5Fu1CvMWVd7h1UqtRCguZp4r/qvGkWYTU/HboUgOWmhpXsaJSbq3TGujUL2X9dxBC84J+WzMCS4BCNFioAJE1zFA8XButpNgclheILtaexrdv5X6Oo1SCF4uoIAC+aKo84zRE6bXK69XGVVGprxdPXvagtLVl43sjTRzLq2pyjeDummTFfaa8y5YtFFRY0UmSQBCCgJwBbEI8A+gNztME3m9Oz9UUixzdUh4iL4QGYA/SfEvTre96r6/Um2sptlIjLWOoS0FMVMWO4PRLiJaMRiaZgfcGYG1Zh8XALKad8s1BoEYzskhCX2EqW1Zt/dHE7HU11yjcAkPcup8UrctYKsXYxRvtqhbYbj6NClLq9JC8la/LyYanvOfubDexKusqoRi7uLyh+Wfnv9BNwj9Jcb0XgCFAvVdhPWdNekm/3gbiwM1u1fHRHmJci5l6LTFfklC+URjym/LNQLCiS8Ej/0o+ckhMI9VdA/f37wCjNEjJKAMhnkJlEso6ggtIbxiaMI0A1rjlAnOhtfP/3PhBPjUrAAA="""
MODEL_URL = "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/9217f5db79a29953eb74d5343926648285ec7e67/qwen2.5-0.5b-instruct-q8_0.gguf?download=true"
MODEL_BYTES = 675710816
MODEL_SHA256 = "ca59ca7f13d0e15a8cfa77bd17e65d24f6844b554a7b6c12e07a5f89ff76844e"
ROOT = Path("/kaggle/working/wave118")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
TARGET = ROOT / "target"
MODEL = ROOT / "qwen2.5-0.5b-instruct-q8_0.gguf"
FINAL_ZIP = Path("/kaggle/working/glcuda-t4-wave118-in-process-stability-results.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)
RESULTS.mkdir(parents=True)

def run(cmd, *, cwd=None, env=None, timeout=14400, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({k: str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       capture_output=True, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if p.stdout:
        print(p.stdout[-12000:], flush=True)
    if p.stderr:
        print(p.stderr[-12000:], flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, p):
    (RESULTS / name).write_text(
        f"RETURN_CODE {p.returncode}\n\nSTDOUT\n{p.stdout}\n\nSTDERR\n{p.stderr}",
        encoding="utf-8",
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def archive():
    if FINAL_ZIP.exists():
        FINAL_ZIP.unlink()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    digest = sha256_file(FINAL_ZIP)
    print("ARCHIVE", FINAL_ZIP, digest, flush=True)
    return digest

def percentile(values, q):
    values = sorted(values)
    x = (len(values) - 1) * q
    lo, hi = math.floor(x), math.ceil(x)
    return values[lo] if lo == hi else values[lo] * (hi - x) + values[hi] * (x - lo)

def bootstrap_ci(values, seed=118, draws=20000):
    rng = random.Random(seed)
    n = len(values)
    medians = [statistics.median(values[rng.randrange(n)] for _ in range(n))
               for _ in range(draws)]
    return [percentile(medians, 0.025), percentile(medians, 0.975)]

phase = "bootstrap"
try:
    embedded = gzip.decompress(base64.b64decode(PATCH_GZIP_B64))
    if hashlib.sha256(embedded).hexdigest() != PATCH_SHA256:
        raise RuntimeError("embedded patch hash mismatch")
    patch_path = RESULTS / "wave118.patch"
    patch_path.write_bytes(embedded)
    (RESULTS / "source.json").write_text(json.dumps({
        "build": BUILD, "base_rev": BASE_REV, "source_rev": SOURCE_REV,
        "patch_sha256": PATCH_SHA256, "patch_bytes": len(embedded),
    }, indent=2), encoding="utf-8")

    gpu = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    fields = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
    if len(fields) < 5 or fields[1] != "Tesla T4" or fields[2] != "7.5":
        raise RuntimeError(f"requires Tesla T4 sm_75, got {fields}")

    phase = "reconstruct"
    clone = run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800)
    save("git-clone.log", clone)
    checkout = run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=600)
    save("git-checkout.log", checkout)
    applied = run(["git", "apply", "--whitespace=error", patch_path], cwd=TREE)
    save("git-apply.log", applied)
    diff = run(["git", "diff", "--check"], cwd=TREE)
    save("git-diff-check.log", diff)

    cargo_candidates = [shutil.which("cargo"), Path.home() / ".cargo/bin/cargo",
                        "/usr/local/cargo/bin/cargo", "/opt/conda/bin/cargo"]
    cargo = next((str(x) for x in cargo_candidates if x and Path(x).is_file()), None)
    cargo_env = {}
    bootstrapped = False
    if cargo is None:
        bootstrapped = True
        rustup_script = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", rustup_script)
        cargo_home = ROOT / "cargo-home"
        rustup_home = ROOT / "rustup-home"
        cargo_env = {"CARGO_HOME": cargo_home, "RUSTUP_HOME": rustup_home}
        install = run(["bash", rustup_script, "-y", "--profile", "minimal",
                       "--default-toolchain", "stable", "--no-modify-path"],
                      env=cargo_env, timeout=1800)
        save("rustup-install.log", install)
        cargo = str(cargo_home / "bin/cargo")
    if not Path(cargo).is_file():
        raise RuntimeError(f"cargo unavailable after bootstrap: {cargo}")
    (RESULTS / "cargo-discovery.json").write_text(json.dumps({
        "selected": cargo, "bootstrapped": bootstrapped,
        "candidates": [str(x) for x in cargo_candidates if x],
    }, indent=2), encoding="utf-8")
    common = {**cargo_env, "CARGO_TARGET_DIR": TARGET, "CUDA_VISIBLE_DEVICES": "0"}

    phase = "host-tests"
    tests = run([cargo, "test", "-p", "glcuda", "--lib", "--locked"], cwd=TREE, env=common)
    save("cargo-lib-tests.log", tests)
    if "67 passed" not in tests.stdout or "0 failed" not in tests.stdout:
        raise RuntimeError("unexpected host test summary")

    phase = "cuda-parity"
    parity = run([cargo, "test", "--release", "-p", "glcuda", "--test", "parity",
                  "--locked", "--", "--nocapture", "--test-threads=1"],
                 cwd=TREE, env=common, check=False)
    save("cargo-cuda-parity.log", parity)
    if parity.returncode or "0 failed" not in parity.stdout:
        raise RuntimeError("CUDA parity failed")

    phase = "compiler-resource"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    resource = run([ptxas, "-v", "-arch=sm_75", TREE / "glcuda/src/kernels/glcuda_sm75.ptx",
                    "-o", ROOT / "wave118.cubin"], check=False)
    save("ptxas-sm75.log", resource)
    if resource.returncode or "spill stores" not in resource.stderr:
        raise RuntimeError("ptxas resource gate failed")

    phase = "model"
    urllib.request.urlretrieve(MODEL_URL, MODEL)
    model_meta = {"bytes": MODEL.stat().st_size, "sha256": sha256_file(MODEL)}
    if model_meta != {"bytes": MODEL_BYTES, "sha256": MODEL_SHA256}:
        raise RuntimeError(f"model identity mismatch: {model_meta}")
    (RESULTS / "model.json").write_text(json.dumps(model_meta, indent=2), encoding="utf-8")

    phase = "build"
    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave118_in_process_stability", "--locked"], cwd=TREE, env=common)
    save("cargo-build.log", build)
    exe = TARGET / "release/examples/wave118_in_process_stability"
    prod_env = {**common, "GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1",
                "GLCUDA_FUSE_Q8_GLUE": "1", "GLCUDA_NTILE128": "1",
                "GLCUDA_BSTAGE": "1", "GLCUDA_GEMM_N16": "1",
                "GLCUDA_GEMM_N16_PREFETCH": "1", "GLCUDA_ATTN_MMA4": "1",
                "GLCUDA_ATTN_MMA4_REGQ": "1", "GLCUDA_ATTN_MMA4_AV": "1"}

    phase = "production"
    records = []
    invocation_rows = {}
    for invocation in ("a", "b"):
        measured = run([exe, MODEL, invocation], cwd=TREE, env=prod_env, check=False)
        save(f"production-{invocation}.log", measured)
        if measured.returncode:
            raise RuntimeError(f"invocation {invocation} failed")
        rows = [json.loads(x) for x in re.findall(
            r"\[wave118-sample\]\s*(\{[^\n]+\})", measured.stdout)]
        if len(rows) != 200 or any(x["prompt_tokens"] != 244 for x in rows):
            raise RuntimeError(f"invocation {invocation} sample contract failed: {len(rows)}")
        invocation_rows[invocation] = rows
        records.extend(rows)

    paired = []
    for invocation, rows in invocation_rows.items():
        for quartet in range(50):
            q = [x for x in rows if x["quartet"] == quartet]
            retained = [x["prefill_ms"] for x in q if x["arm"] == "retained"]
            candidate = [x["prefill_ms"] for x in q if x["arm"] == "candidate"]
            if len(retained) != 2 or len(candidate) != 2:
                raise RuntimeError("ABBA quartet contract failed")
            paired.append({"invocation": invocation, "quartet": quartet,
                           "log_speedup": math.log(statistics.mean(retained) /
                                                   statistics.mean(candidate))})
    logs = [x["log_speedup"] for x in paired]
    ci = bootstrap_ci(logs)
    by_invocation = {name: math.exp(statistics.median(
        x["log_speedup"] for x in paired if x["invocation"] == name))
        for name in ("a", "b")}
    summary = {
        "wave": 118, "gpu": fields, "model": model_meta,
        "samples_per_arm_per_invocation": 100, "total_samples": len(records),
        "oracle_matches": f"{len(records)}/{len(records)}",
        "median_speedup": math.exp(statistics.median(logs)),
        "p10_speedup": math.exp(percentile(logs, 0.10)),
        "p90_speedup": math.exp(percentile(logs, 0.90)),
        "bootstrap_ci95_speedup": [math.exp(ci[0]), math.exp(ci[1])],
        "invocation_median_speedup": by_invocation,
        "diagnostic_pass": all(x > 1.0 for x in by_invocation.values()) and math.exp(ci[0]) > 0.99,
        "retention_authority": False, "target_15000_tps_achieved": False,
    }
    (RESULTS / "production-records.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
    (RESULTS / "paired-log-speedups.json").write_text(json.dumps(paired, indent=2), encoding="utf-8")
    (RESULTS / "wave118-stability.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("WAVE118_RESULT", json.dumps(summary, indent=2), flush=True)
    archive()
except Exception:
    (RESULTS / "FAILED.json").write_text(json.dumps({
        "phase": phase, "traceback": traceback.format_exc()}, indent=2), encoding="utf-8")
    archive()
    raise
